# Feature Engineering and Initial ML Model Testing

This script:
1. Loads cleaned Yelp review datasets from 'cleaned-data/'.
2. Converts text into TF-IDF features with n-grams.
3. Trains a basic Logistic Regression model for sentiment prediction.
4. Outputs basic accuracy metrics.

In [1]:
import glob
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
folder = "cleaned-data/"
csv_files = glob.glob(os.path.join(folder, "*.csv"))

dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    df['state'] = os.path.splitext(os.path.basename(file))[0] 
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data = data.dropna()
print(f"Loaded {len(data)} reviews from {len(dfs)} states.")

Loaded 5222860 reviews from 20 states.


In [4]:
# creating sentiment score for easier classification
data['sentiment'] = data['stars'].apply(lambda x: 0 if x < 3 else (1 if x == 3 else 2))
data.sample(5)

,stars,text,review_length,num_exclamations,num_caps_words,clean_text,state,sentiment
2477110,3.0,"I love this place, I think it's a great concep...",24,0,3,love place think great concept everytime go tw...,FL,1
2056911,4.0,Went last night and it was a lot of fun. They ...,47,0,1,went last night lot fun free drinks free get l...,FL,2
4783970,4.0,The best fried oysters I've ever had ever! Alt...,61,1,0,best fried oysters ive ever ever although serv...,LA,2
4889193,5.0,Houston's on St. Charles is a restaurant you h...,48,0,0,houstons st charles restaurant try hawaiian st...,LA,2
4938029,3.0,"We got alligator, Mac n cheese, fried liver on...",13,0,1,got alligator mac n cheese fried liver toast ok,LA,1


In [8]:
sample = data.sample(n=500000) # only using 500,000 data points for the sake of computation time (instead of 5,000,000) (arbitrary)

X = sample['clean_text']
# y = sample['stars'] # 1-5 RRP
y = sample['sentiment'] # pos/neg/neutral
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

# TF-IDF with n-grams
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=10000)  # unigrams + bigrams
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Initial ML model: Logistic Regression 
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8757

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.86      0.84     10223
           1       0.58      0.36      0.44      5677
           2       0.92      0.97      0.94     34100

    accuracy                           0.88     50000
   macro avg       0.78      0.73      0.74     50000
weighted avg       0.86      0.88      0.87     50000

